# Ranking Metrics and Evaluation Cohorts

Evaluate candidate ordering within each logged impression, then aggregate over eligible impressions. Report metric-specific cohort sizes because undefined AUC and missing cohorts need not share the same denominator.

## Reciprocal rank

For relevant ranks $r_1,\ldots,r_m$, the MLP-style impression metric is $RR_{mean}=\frac1m\sum_{j=1}^{m}1/r_j$. The hybrid/NRMS first-hit definition is $RR_{first}=1/\min_j r_j$. These coincide with one clicked candidate but differ with multiple clicks. Retain the named source definition when comparing models.

## Normalized discounted cumulative gain

For binary label $y_r$ at predicted rank $r$,

$$DCG@k=\sum_{r=1}^{\min(k,n)}\frac{y_r}{\log_2(r+1)},\qquad nDCG@k=\frac{DCG@k}{IDCG@k}.$$

IDCG uses the same labels in ideal order. Report cutoffs 5 and 10. The shared primitive returns zero when ideal gain is zero; source workflow filtering determines whether such impressions enter aggregate results.

## AUC and ties

The pairwise interpretation is the probability that a clicked candidate outranks an unclicked candidate, with half credit for ties in average-rank implementations. Source evaluator conventions differ for one-class impressions and score ties; preserve those policies and identify the evaluator used. Ranking metrics must not sort using labels to favor relevant tied candidates.

## Cohorts and evidence

Cold users have history length 0, sparse users 1–4 and warm users at least 5 at impression time. Article-cold flags retain their source click-history window and selection policy; an article flag is not a mutually exclusive user segment. MLP cold-article evaluation retains non-clicked candidates and clicked cold candidates while removing clicked warm candidates. The shared hybrid evaluator instead selects complete impressions containing a clicked cold item. Do not combine these populations under an unlabeled single estimate.

The smoothed-CTR baseline uses the MLP evaluator to match that fallback comparison. Scores alone do not establish calibrated real-world CTR, counterfactual uplift or engagement on unobserved slates.


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "utils" / "data_utils.py").is_file()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.data_utils import project_paths, load_preprocessed, load_table, save_table

if os.environ.get("BT4222_USE_DRIVE") == "1":
    from google.colab import drive

    drive.mount("/content/drive")

PATHS = project_paths(PROJECT_ROOT)


In [ ]:
import numpy as np
from utils.metrics import ndcg_at_k, history_groups

labels = np.array([1, 0, 1, 0])
print({f"nDCG@{k}": ndcg_at_k(labels, labels, k) for k in [5, 10]})
print(history_groups([0, 1, 4, 5]))
